# 1. Experiment 2 — Damage Detection Improvement

# Experiment 2 — Damage Detection Improvement

This notebook implements Experiment 2 for the Qaddir vehicle damage detection model.

The experiment builds on the YOLO11n baseline model and evaluates whether
stronger data augmentation and an extended training schedule can improve
damage detection performance.

The dataset, class taxonomy, and Train/Validation/Test splits remain unchanged
from the baseline experiment.

# 2. Objective

The objective of Experiment 2 is to improve the baseline damage detection
performance through controlled changes to data augmentation and training
hyperparameters.

The experiment will be compared against Experiment 1 (Baseline).

Primary comparison metrics:

- Precision
- Recall
- mAP50
- mAP50-95

The validation set is kept unchanged to ensure a fair comparison.

# 3. Import Libraries

In [4]:
import Path
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu126
CUDA available: True
GPU: NVIDIA GeForce MX450


In [20]:
import matplotlib
print("Matplotlib version:", matplotlib.__version__)

from matplotlib.backends import backend_registry
print("Matplotlib backend registry: OK")

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

# 4. Set Project Paths

In [8]:
PROJECT_ROOT = Path(
    r"C:\Users\xlosn\qaddir-vehicle-damage-assessment"
)

YOLO_DIR = (
    PROJECT_ROOT
    / "data"
    / "CarDD_release"
    / "CarDD_YOLO"
)

DATA_YAML = YOLO_DIR / "data.yaml"

RUNS_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "runs"
    / "detect"
    / "runs"
    / "cardd"
)

EXPERIMENT_NAME = "experiment_2_augmented"

EXPERIMENT_DIR = RUNS_DIR / EXPERIMENT_NAME

print("Project:", PROJECT_ROOT)
print("Dataset:", YOLO_DIR)
print("Data YAML:", DATA_YAML)
print("Experiment output:", EXPERIMENT_DIR)

print("\nData YAML exists:", DATA_YAML.exists())
print("YOLO dataset exists:", YOLO_DIR.exists())

Project: C:\Users\xlosn\qaddir-vehicle-damage-assessment
Dataset: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO
Data YAML: C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO\data.yaml
Experiment output: C:\Users\xlosn\qaddir-vehicle-damage-assessment\notebooks\runs\detect\runs\cardd\experiment_2_augmented

Data YAML exists: True
YOLO dataset exists: True


# 5. Verify Dataset Configuration

In [9]:
print(DATA_YAML.read_text(encoding="utf-8"))

path: C:/Users/xlosn/qaddir-vehicle-damage-assessment/data/CarDD_release/CarDD_YOLO
train: images/train
val: images/val
test: images/test

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat



# 6. Reproducibility
Set Random Seeds
A fixed random seed is used to improve reproducibility between experiments.

In [10]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


# 7. Baseline Reference Metrics
Experiment 1 achieved the following validation performance:

- Precision: 0.625
- Recall: 0.576
- mAP50: 0.587
- mAP50-95: 0.462

These values are used as the reference point for Experiment 2.


In [11]:
baseline_metrics = {
    "Precision": 0.625,
    "Recall": 0.576,
    "mAP50": 0.587,
    "mAP50-95": 0.462
}

baseline_df = pd.DataFrame(
    [baseline_metrics],
    index=["Experiment 1 - Baseline"]
)

baseline_df

,Precision,Recall,mAP50,mAP50-95
Experiment 1 - Baseline,0.625,0.576,0.587,0.462


# 8. Experiment 2 Configuration

Experiment 2 keeps the following baseline conditions unchanged:

- Model: YOLO11n
- Image size: 640
- Batch size: 4
- Device: GPU 0
- Dataset: CarDD YOLO
- Validation split: unchanged
- AMP: disabled due to baseline hardware limitations
- Random seed: 0

The following changes are introduced:

- Training duration increased from 10 to 15 epochs
- Rotation augmentation enabled
- Slightly stronger translation
- Increased scaling variation
- MixUp augmentation enabled
- Copy-Paste augmentation enabled

The purpose is to improve robustness to variations in vehicle appearance,
damage position, scale, and visual conditions.

In [12]:
EXPERIMENT_CONFIG = {
    "model": "yolo11n.pt",
    "epochs": 15,
    "imgsz": 640,
    "batch": 4,
    "device": 0,
    "workers": 2,
    "amp": False,
    "seed": 0,

    # Augmentation
    "degrees": 5.0,
    "translate": 0.15,
    "scale": 0.6,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.10,
    "copy_paste": 0.10,

    # Training
    "patience": 15,
    "project": str(RUNS_DIR),
    "name": EXPERIMENT_NAME,
    "exist_ok": True
}

for key, value in EXPERIMENT_CONFIG.items():
    print(f"{key}: {value}")

model: yolo11n.pt
epochs: 15
imgsz: 640
batch: 4
device: 0
workers: 2
amp: False
seed: 0
degrees: 5.0
translate: 0.15
scale: 0.6
fliplr: 0.5
flipud: 0.0
mosaic: 1.0
mixup: 0.1
copy_paste: 0.1
patience: 15
project: C:\Users\xlosn\qaddir-vehicle-damage-assessment\notebooks\runs\detect\runs\cardd
name: experiment_2_augmented
exist_ok: True


# 9. Save Experiment Configuration

In [13]:
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

config_file = EXPERIMENT_DIR / "experiment_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=4)

print("Configuration saved to:")
print(config_file)

Configuration saved to:
C:\Users\xlosn\qaddir-vehicle-damage-assessment\notebooks\runs\detect\runs\cardd\experiment_2_augmented\experiment_config.json


# 10. Load YOLO11n

In [14]:
model = YOLO("yolo11n.pt")

print("YOLO11n loaded successfully.")

YOLO11n loaded successfully.


# 11. Run Experiment 2 Training

In [16]:
results = model.train(
    data=str(DATA_YAML),

    epochs=15,
    imgsz=640,
    batch=4,

    device=0,
    workers=2,
    amp=False,

    seed=0,

    # Augmentation
    degrees=5.0,
    translate=0.15,
    scale=0.6,
    fliplr=0.5,
    flipud=0.0,

    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.10,

    patience=15,

    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.148 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.144  Python-3.11.9 torch-2.14.0+cu126 CUDA:0 (NVIDIA GeForce MX450, 2048MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\xlosn\qaddir-vehicle-damage-assessment\data\CarDD_release\CarDD_YOLO\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.0

RuntimeError: Dataset 'C://Users/xlosn/qaddir-vehicle-damage-assessment/data/CarDD_release/CarDD_YOLO/data.yaml' error  No module named 'matplotlib.backends.registry'

# 12. Locate Best Model

In [ ]:
BEST_MODEL = EXPERIMENT_DIR / "weights" / "best.pt"
LAST_MODEL = EXPERIMENT_DIR / "weights" / "last.pt"

print("Best model:")
print(BEST_MODEL)

print("\nExists:", BEST_MODEL.exists())

print("\nLast model:")
print(LAST_MODEL)

print("Exists:", LAST_MODEL.exists())

# 13. Validate Experiment 2
The best checkpoint from Experiment 2 is evaluated on the unchanged
validation set.

The test set is not used at this stage.

In [ ]:
best_model = YOLO(str(BEST_MODEL))

validation_results = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    amp=False
)

# 14. Extract Overall Metrics

In [ ]:
experiment_2_metrics = {
    "Precision": float(validation_results.box.mp),
    "Recall": float(validation_results.box.mr),
    "mAP50": float(validation_results.box.map50),
    "mAP50-95": float(validation_results.box.map)
}

experiment_2_df = pd.DataFrame(
    [experiment_2_metrics],
    index=["Experiment 2 - Augmentation"]
)

experiment_2_df

# 15. Baseline vs Experiment 2 Comparison

In [ ]:
comparison_df = pd.concat([
    baseline_df,
    experiment_2_df
])

comparison_df

# 16. Calculate Improvement

In [ ]:
improvement = {}

for metric in baseline_metrics:
    baseline_value = baseline_metrics[metric]
    experiment_value = experiment_2_metrics[metric]

    absolute_change = experiment_value - baseline_value

    if baseline_value != 0:
        percentage_change = (
            absolute_change / baseline_value
        ) * 100
    else:
        percentage_change = 0

    improvement[metric] = {
        "Baseline": baseline_value,
        "Experiment 2": experiment_value,
        "Absolute Change": absolute_change,
        "Percentage Change": percentage_change
    }

improvement_df = pd.DataFrame(improvement).T

improvement_df

# 17. Per-Class Evaluation
Per-class metrics are examined to determine whether Experiment 2 improves
performance across individual damage categories, particularly difficult
classes such as Crack, Dent, and Scratch.

In [ ]:
class_names = [
    "dent",
    "scratch",
    "crack",
    "glass shatter",
    "lamp broken",
    "tire flat"
]

per_class_results = []

for class_id, class_name in enumerate(class_names):

    precision = float(validation_results.box.p[class_id])
    recall = float(validation_results.box.r[class_id])
    map50 = float(validation_results.box.ap50[class_id])
    map5095 = float(validation_results.box.ap[class_id])

    per_class_results.append({
        "Class": class_name,
        "Precision": precision,
        "Recall": recall,
        "mAP50": map50,
        "mAP50-95": map5095
    })

per_class_df = pd.DataFrame(per_class_results)

per_class_df

# 18. Save Metrics

In [ ]:
comparison_file = EXPERIMENT_DIR / "experiment_2_comparison.csv"
per_class_file = EXPERIMENT_DIR / "experiment_2_per_class_metrics.csv"

comparison_df.to_csv(comparison_file)
per_class_df.to_csv(per_class_file, index=False)

print("Saved:")
print(comparison_file)
print(per_class_file)

# 19. Plot Training Results
The training curves generated by Ultralytics are inspected to understand
whether the model continued improving during training and whether signs
of overfitting are present.

In [ ]:
from IPython.display import display
from PIL import Image

results_plot = EXPERIMENT_DIR / "results.png"

print("Results plot:")
print(results_plot)
print("Exists:", results_plot.exists())

if results_plot.exists():
    display(Image.open(results_plot))

# 21. Experiment 2 Conclusion

Experiment 2 was conducted using the same CarDD dataset and validation split
as the baseline experiment.

The experiment introduced stronger augmentation and an extended training
schedule while keeping the YOLO11n architecture unchanged.

The final conclusion will be based on the validation metrics obtained from
Experiment 2.

Experiment 2 will be considered an improvement only if it demonstrates
meaningful improvement over the baseline without unacceptable degradation
in important damage classes.

# 22. Save Final Experiment Summary

In [ ]:
summary_file = EXPERIMENT_DIR / "experiment_2_summary.txt"

summary_text = f"""
Qaddir - Experiment 2 Summary
==============================

Model:
YOLO11n

Dataset:
CarDD

Training:
Epochs: 15
Image Size: 640
Batch Size: 4
Seed: 0
AMP: False

Augmentation:
Degrees: 5.0
Translate: 0.15
Scale: 0.6
Flip LR: 0.5
Flip UD: 0.0
Mosaic: 1.0
MixUp: 0.10
Copy-Paste: 0.10

Baseline Metrics:
Precision: {baseline_metrics['Precision']}
Recall: {baseline_metrics['Recall']}
mAP50: {baseline_metrics['mAP50']}
mAP50-95: {baseline_metrics['mAP50-95']}

Experiment 2 Metrics:
Precision: {experiment_2_metrics['Precision']:.4f}
Recall: {experiment_2_metrics['Recall']:.4f}
mAP50: {experiment_2_metrics['mAP50']:.4f}
mAP50-95: {experiment_2_metrics['mAP50-95']:.4f}
"""

summary_file.write_text(summary_text, encoding="utf-8")

print(summary_file)